<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Review generation

## Set up

In [ ]:
# @title Dependencies

# Installation
!pip install pandas pyarrow -q
!pip install numpy -q
!pip install tqdm -q
!pip install openai -q
!pip install groq -q

# First-party installations
import itertools
import re
import json
import hashlib
import random
import time
from typing import Dict, Any, Tuple, Optional, Literal, List
from dataclasses import dataclass
import os
import asyncio

# Third-party installations
from google.colab import drive
import google.colab.userdata
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import AsyncOpenAI
from openai import APIError
from groq import Groq
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

In [ ]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews/generated")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "results.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "results.jsonl")

# Define the full log path (final Parquet output)
LOG_PATH = os.path.join(RAW_DIR, "logs.parquet")
# Define the full log path for intermediate JSONL storage
LOG_JSON_PATH = os.path.join(RAW_DIR, "logs.jsonl")

In [ ]:
# @title Setup definitions

### --- Settings --- ###

# Define mode
SIMULATION_ACTIVE = True

# Define error probability for note taking or review generation failure
ERROR_PROB = 0.1

# Define number of paper-parameter combos to process in parallel
NUM_PARALLEL_BATCH_SIZE = 10

# Define subset configuration
PAPER_SUBSETTING_ACTIVE = False
NUM_SUBSET = 10 # Number of papers per configuration combo
CONFIG_SUBSETTING_ACTIVE = False
CONFIG_SUBSET_MODELS = [] # Entries subset these dimensions to what is given; Else no subsetting is applied to them
CONFIG_SUBSET_NOTE_TAKING_OPTIONS = []
CONFIG_SUBSET_REASONING_EFFORTS = []
CONFIG_SUBSET_CHAIN_OF_THOUGHTS = []
CONFIG_SUBSET_TEMPS = []
CONFIG_NUM_SUBSET_PER_MODEL = 1 # Number of parameter combinations to keep PER MODEL after filtering

### --- Parameters --- ###

# Define OpenAI models
OPENAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-11-20",
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"
]

# Define Groq model
GROQ_MODELS = [
    "llama-3.3-70b-versatile"
]

# Define Gemini model
GEMINI_MODELS = [
    "gemini-3-flash-preview"
]

# Define reasoning models
REASONING_MODELS = {
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gemini-3-flash-preview"
}

# Define temperature parameters (as used by Cummins 2025)
TEMPS = [0.0, 0.5, 1.0, 1.5]

# Define reasoning parameters (as used by Cummins 2025)
REASONING = ["low", "high"]

# Define prompt style (CoT) (as used by Li et al. 2025)
CoT = [
    "",
    "Explain your thought process step by step."
]

# Define note taking (adopted from Robertson and Kayejo (2025)) (plus additional instruction to skip note taking at all)
NOTE_TAKING = [
    "Faithful",
    "Turned off"
]

# Create data class for parameter combinations
@dataclass
class ParamCombo:
    model: str
    temperature: Optional[float]
    reasoning_effort: Optional[Literal[*REASONING]]
    chain_of_thought: Literal[*CoT]
    note_taking: Literal[*NOTE_TAKING]

### --- Content --- ####

# Define target string to remove metadata from the papers full text
HEADER_TO_REMOVE = "GROBID - A machine learning software for extracting information from scholarly documents"

# Define Note taking string in case it is skipped
NOTES_PLACEHOLDER = "NOTES_SKIPPED"

# Define note taking strategies, taken from Robertson and Koyejo (2025)
NOTE_TAKING_FAITHFUL = f"Take detailed, accurate notes on the paper for an ICLR style review. Focus on precisely capturing the actual methods, results, and contributions without any distortion. Just output the notes."
NOTE_TAKING_PROMPT = {"Faithful": NOTE_TAKING_FAITHFUL}

# Define review generation instruction, adopted from Robertson and Koyejo (2025)
REVIEW_GENERATION = f"""

Create an ICLR-style review following this specific structure:

# Summary Of The Paper
Summarize the paper’s main contributions, methodology, and findings.

# Strength And Weaknesses
Analyze the paper’s contributions based on your notes.

# Clarity, Quality, Novelty And Reproducibility
Evaluate based on your notes.

# Summary Of The Review
Provide a 2-3 sentence distillation of your overall assessment.

# Correctness
Rate on a scale of 1-5.

# Technical Novelty And Significance
Rate on a scale of 1-5.

# Empirical Novelty And Significance
Rate on a scale of 1-5.

Maintain a professional tone throughout. Base your review entirely on your reading notes.

"""

# Define error strings for failed note taking
NOTE_TAKING_FAILURE = "ERROR: NOTE TAKING NOT SUCCESSFUL"

# Define error strings for failed review generation
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"

# Define the Pydantic schema for the review output
class ReviewOutput(BaseModel):
    summary_of_paper: str = Field(..., description="Summary of the paper’s main contributions, methodology, and findings.")
    strengths: List[str] = Field(..., description="Analysis of the paper’s strengths in bulletpoints")
    weaknesses: List[str] = Field(..., description="Analysis of the paper’s weaknesses in bulletpoints")
    clarity_quality_novelty_reproducibility: List[str] = Field(..., description="Analysis of the paper's clarity, quality, novelty, and reproducibility in bulletpoints")
    summary_of_review: str = Field(..., description="A 2-3 sentence distillation of the overall assessment.")
    correctness_rating: int = Field(..., ge=1, le=5, description="Rating on a scale of 1-5 for correctness.")
    technical_novelty_significance_rating: int = Field(..., ge=1, le=5, description="Rating on a scale of 1-5 for technical novelty and significance.")
    empirical_novelty_significance_rating: int = Field(..., ge=1, le=5, description="Rating on a scale of 1-5 for empirical novelty and significance.")

### --- API/Client --- ###

# Define client call parameters
MAX_RETRIES = 3 # Random
RETRY_DELAY = 1.5 # Random
SEED_VALUE = 45 # Random
MAX_TOKENS = 2000 # As Robertson and Kayejo (2025) (in their repo)

# Set OpenAI API-key
OPENAI_KEY = google.colab.userdata.get('OPENAI_API_KEY')

# Set Groq API-key
GROQ_KEY = google.colab.userdata.get('GROQ_API_KEY')

# Set Gemini API-key
GEMINI_API_KEY = google.colab.userdata.get('GEMINI_API_KEY')

# Set Groq client URL
GROQ_URL ="https://api.groq.com/openai/v1"

In [ ]:
# @title Data import

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "dataset_paper.parquet"))

# Select target columns
paper_content = dataset_paper[['paper_id', 'abstract', 'parsed_text']].copy()

# Remove the HEADER_TO_REMOVE string from the 'parsed_text' column
paper_content['parsed_text'] = paper_content['parsed_text'].str.replace(HEADER_TO_REMOVE, '', regex=False).str.strip()

# Check and display paper_content
display(paper_content.shape)
display(paper_content.head(3))

# Prepare Dict for later applications
targets = paper_content.to_dict('records')

In [ ]:
# @title Subset paper content (if active)

if PAPER_SUBSETTING_ACTIVE:
    # Subset to the first NUM_SUBSET rows
    paper_content = paper_content.head(NUM_SUBSET)
    print(f"Paper content subsetted to {NUM_SUBSET} rows.")

    # Check and display subset
    display(paper_content.shape)
    display(paper_content.head(3))

    # Update targets after subsetting
    targets = paper_content.to_dict('records')
else:
    print("Paper content subsetting is inactive.")

## Define

In [ ]:
# @title Generation client

class AsyncLLMClient:
    def __init__(self, seed: int, simulate: bool = True):
        """Initialize AsyncLLMClient with defaults"""
        self.simulate = simulate
        self.rng = random.Random(seed)

        # Set placeholders for real clients when simulate=False:
        self._openai = None
        self._groq_openai_client = None
        self._gemini_client = None

    def _maybe_init_clients(self):
        """Start simulation/API clients"""
        if self.simulate:
            return
        # Initialize OpenAI client (AsyncOpenAI)
        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = AsyncOpenAI(api_key=OPENAI_KEY)

        # Initialize Groq client using OpenAI client interface (AsyncOpenAI)
        if self._groq_openai_client is None and "GROQ_KEY" in globals():
            self._groq_openai_client = AsyncOpenAI(api_key=GROQ_KEY, base_url=GROQ_URL)

        # Configure Gemini API globally and instantiate client
        if self._gemini_client is None and "GEMINI_API_KEY" in globals():
            self._gemini_client = genai.Client(api_key=GEMINI_API_KEY)

    async def call(
        self,
        prompt_content: str,
        model: str,
        temperature: Optional[float],
        reasoning_effort: Optional[str],
        chain_of_thought: str,
        note_taking: str,
        max_retries: float = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
        max_tokens: int = MAX_TOKENS,
    ) -> Tuple[str, str]:
        """
        Defines the API calls / simulations according to the configuration setup.
        Returns: notes, review.
        """

        self._maybe_init_clients()

        # --- Step 1: Generate notes ---

        # Initialize note placeholder for review generation step
        content_for_review_step = None
        # Initialize note placeholder for saving
        notes_to_save = None

        if note_taking == "Turned off":
            content_for_review_step = prompt_content  # Use full paper content for the review step
            notes_to_save = NOTES_PLACEHOLDER # Set placeholder for final save (to reduce memory)
        else:
            # Initialize generated_notes for this block
            generated_notes_from_api = None

            # Construct final prompt basis for note taking
            note_taking_instruction = NOTE_TAKING_PROMPT.get(note_taking, note_taking)
            final_note_taking_prompt = f"{note_taking_instruction}\n\n{prompt_content}{note_taking_instruction}\n\n".strip()

            # Add optional chain of thought instruction
            if chain_of_thought:
                final_note_taking_prompt += f" {chain_of_thought}"

            # Set provider for note-taking model
            provider_notes = "openai"
            if model in GROQ_MODELS:
                provider_notes = "groq"
            elif model in GEMINI_MODELS:
                provider_notes = "gemini"

            for attempt in range(1, max_retries + 1):
                try:
                    # Simulate artificial note
                    if self.simulate:
                        if self.rng.uniform(0, 1) < ERROR_PROB:
                            generated_notes_from_api = NOTE_TAKING_FAILURE
                        else:
                            generated_notes_from_api = (
                                f"This is a simulated note based on: Model='{model}', Temp='{temperature}', "
                                f"Effort='{reasoning_effort}', CoT='{chain_of_thought}', Notes='{note_taking}'. "
                            )
                        break
                    # Call API
                    else:
                        # For OpenAI (reasoning) models
                        if provider_notes == "openai" and model in REASONING_MODELS:
                            kwargs = {}
                            # Set hyperparameter
                            kwargs["reasoning"] = {"effort": reasoning_effort}
                            # Set max_output_tokens
                            kwargs["max_output_tokens"] = MAX_TOKENS

                            # Specify the user message
                            msgs = [{"role": "user", "content": final_note_taking_prompt}]

                            # Specify the client to use
                            client_to_use = self._openai

                            # Call client and return the notes
                            resp = await client_to_use.responses.create(
                                model=model,
                                input=msgs,
                                **kwargs,
                            )
                            generated_notes_from_api = resp.output_text
                            break

                        # For OpenAI (non-reasoning) and Groq models
                        if (provider_notes == "openai" and model not in REASONING_MODELS) or provider_notes == "groq":
                            kwargs = {}
                            # Set hyperparameter
                            if temperature is not None:
                                kwargs["temperature"] = float(temperature)
                            # Set max_tokens
                            kwargs["max_tokens"] = MAX_TOKENS

                            # Specify the user message
                            msgs = [{"role": "user", "content": final_note_taking_prompt}]

                            # Specify the client to use
                            client_to_use = self._groq_openai_client if provider_notes == "groq" else self._openai

                            # Call client and return the notes (use completions.create cause notes need to structured output)
                            resp = await client_to_use.chat.completions.create(
                                model=model,
                                messages=msgs,
                                **kwargs,
                            )
                            # Extract the notes
                            generated_notes_from_api = resp.choices[0].message.content or ""
                            break

                        # For Gemini models
                        elif provider_notes == "gemini":
                            # Set hyperparameter
                            gen_content_config_args = {}
                            gen_content_config_args["thinking_config"] = types.ThinkingConfig(thinking_level=reasoning_effort)
                            # Set max_tokens
                            gen_content_config_args["max_output_tokens"] = MAX_TOKENS

                            # Call client and return the notes
                            resp = await asyncio.to_thread(
                                self._gemini_client.models.generate_content,
                                model=model,
                                contents=final_note_taking_prompt,
                                config=types.GenerateContentConfig(**gen_content_config_args)
                            )
                            # Extract the notes
                            generated_notes_from_api = resp.text
                            break

                # Raise error flag in case of failure
                except Exception as e:
                    print(f"[LLM ERROR - Note Taking] provider={provider_notes} model={model} attempt={attempt} -> {e}", flush=True)
                    if attempt == max_retries:
                        # Return failure string across both outputs
                        return NOTE_TAKING_FAILURE, REVIEW_GENERATION_FAILURE
                    await asyncio.sleep(retry_delay)

            # Set placeholder strings in case the API failed to generate notes
            if generated_notes_from_api is None or generated_notes_from_api == NOTE_TAKING_FAILURE:
                notes_to_save = NOTE_TAKING_FAILURE
                content_for_review_step = NOTE_TAKING_FAILURE
            else:
                notes_to_save = generated_notes_from_api
                content_for_review_step = generated_notes_from_api

        # Skip review generation step if notes generation failed
        if notes_to_save == NOTE_TAKING_FAILURE:
            return notes_to_save, REVIEW_GENERATION_FAILURE

        # --- Step 2: Generate the main review ---

        # Construct final prompt basis
        final_review_prompt = f"{REVIEW_GENERATION}\n\n{content_for_review_step}{REVIEW_GENERATION}\n\n".strip()

        # Add optional chain of thought instruction
        if chain_of_thought:
            final_review_prompt += f" {chain_of_thought}"

        # Set provider for review model
        provider_review = "openai"
        if model in GROQ_MODELS:
            provider_review = "groq"
        elif model in GEMINI_MODELS:
            provider_review = "gemini"

        generated_review = None # Initialize generated_review for the loop

        for attempt in range(1, max_retries + 1):
            try:
                # Simulate artificial review
                if self.simulate:
                    if self.rng.uniform(0, 1) < ERROR_PROB:
                        generated_review = REVIEW_GENERATION_FAILURE
                    else:
                        # Simulate a structured review that needs to be stringified
                        sim_review_dict = {
                            "summary_of_paper": f"Simulated summary for {model}",
                            "strengths": [f"Simulated strength 1 for {model}", f"Simulated strength 2 for {model}"],
                            "weaknesses": [f"Simulated weakness 1 for {model}", f"Simulated weakness 2 for {model}"],
                            "clarity_quality_novelty_reproducibility": [f"Simulated clarity for {model}", f"Simulated quality for {model}"],
                            "summary_of_review": f"Simulated overall assessment for {model}",
                            "correctness_rating": self.rng.randint(1, 5),
                            "technical_novelty_significance_rating": self.rng.randint(1, 5),
                            "empirical_novelty_significance_rating": self.rng.randint(1, 5)
                        }
                        generated_review = json.dumps(sim_review_dict)
                    return notes_to_save, generated_review
                # Call API
                else:
                    # For OpenAI (reasoning) models
                    if provider_review == "openai" and model in REASONING_MODELS:
                        kwargs = {}
                        # Set hyperparameter
                        kwargs["reasoning"] = {"effort": reasoning_effort}
                        # Set max_output_tokens
                        kwargs["max_output_tokens"] = MAX_TOKENS

                        # Specify the user message
                        msgs = [{"role": "system", "content": "You review a paper based on notes."},
                                {
                                  "role": "user",
                                  "content": f"{final_review_prompt}.",
                                 }
                               ]

                        # Specify the client to use
                        client_to_use = self._openai

                        # Call client and return the review
                        resp = await client_to_use.responses.parse(
                            model=model,
                            input=msgs,
                            text_format=ReviewOutput,
                            **kwargs
                        )
                        # Extract the review as a dictionary, then convert to JSON string
                        generated_review = json.dumps(resp.output_parsed.model_dump())

                    # For OpenAI (non-reasoning) and Groq models
                    elif (provider_review == "openai" and model not in REASONING_MODELS) or provider_review == "groq":
                        kwargs = {}
                        # Set hyperparameter
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)
                        # Add max_tokens directly for these models
                        kwargs["max_tokens"] = MAX_TOKENS

                        # Define response_format for explicit JSON output
                        response_format = {"type": "json_object"}

                        # Construct the system instruction with the JSON schema
                        system_instruction = f"""You are an expert reviewer. Your task is to generate an ICLR-style review based on the provided notes and follow the specific JSON format.
                        Respond only with JSON strictly adhering to the schema:
                        {json.dumps(ReviewOutput.model_json_schema(), indent=2)}
                        """

                        # Specify the user message
                        msgs = [
                            {"role": "system", "content": system_instruction},
                            {"role": "user", "content": final_review_prompt}
                        ]

                        # Specify the client to use
                        client_to_use = self._openai if provider_review == "openai" else self._groq_openai_client

                        # Call client and return the review
                        resp = await client_to_use.chat.completions.create(
                            model=model,
                            messages=msgs,
                            response_format=response_format,
                            **kwargs,
                        )

                        # Extract the content, which is a JSON string
                        generated_review = resp.choices[0].message.content or "{}"

                    # For Gemini models
                    elif provider_review == "gemini":

                        # Set and populate hyperparameter
                        gen_content_config_args = {}
                        gen_content_config_args["thinking_config"] = types.ThinkingConfig(thinking_level=reasoning_effort)
                        gen_content_config_args["response_mime_type"] = "application/json"
                        # Add max_output_tokens
                        gen_content_config_args["max_output_tokens"] = MAX_TOKENS

                        # Construct the system instruction with the JSON schema
                        system_instruction = f"""You are an expert reviewer. Your task is to generate an ICLR-style review based on the provided notes and follow the specific JSON format.
                        Respond only with JSON strictly adhering to the schema:
                        {json.dumps(ReviewOutput.model_json_schema(), indent=2)}
                        """

                        # Combine system instruction and final review prompt
                        combined_prompt = f"{system_instruction}\n\n{final_review_prompt}"

                        # Call client and return the review
                        resp = await asyncio.to_thread(
                            self._gemini_client.models.generate_content,
                            model=model,
                            contents=combined_prompt,
                            config=types.GenerateContentConfig(**gen_content_config_args)
                        )
                        # Extract the review
                        generated_review = resp.text

                    break # Break from the retry loop on success

            # Raise error flag in case of failure
            except Exception as e:
                print(f"[LLM ERROR - Review Generation] provider={provider_review} model={model} attempt={attempt} -> {e}", flush=True)
                if attempt == max_retries:
                    # Return notes and failure message for review generation
                    return notes_to_save, REVIEW_GENERATION_FAILURE
                await asyncio.sleep(retry_delay)

        # If generated_review is still None after retries
        if generated_review is None:
             return notes_to_save, REVIEW_GENERATION_FAILURE

        return notes_to_save, generated_review

In [ ]:
#  @title Combo run

async def run_parameter_combo(
    combo: ParamCombo,
    review_target: Dict,
    client: Optional['AsyncLLMClient'] = None,
) -> Dict:
    """
    Sets id, content and parameters for the client call.
    Calls helper function to print the information.
    Calls the client and returns the results as a dictionary.
    """
    # If client specified then use client, otherwise fake client for test
    client = client or AsyncLLMClient(simulate=True)

    # Extract the current content
    paper_content = review_target['parsed_text']

    # Extract the current id
    doc_id = review_target["paper_id"]

    # Initialize a dictionary for the current id and parameters
    current_paper_record = {
        "paper_id": doc_id,
        "model": combo.model,
        "temperature": combo.temperature,
        "reasoning_effort": combo.reasoning_effort,
        "chain_of_thought": combo.chain_of_thought,
        "note_taking": combo.note_taking,
        "run_signature": RUN_SIGNATURE
    }

    # Include informative output message
    log_call(doc_id, combo)

    # Parse current content and parameters to the client
    notes, review = await client.call(
        model=combo.model,
        prompt_content=paper_content,
        temperature=(combo.temperature if combo.model not in REASONING_MODELS else None),
        reasoning_effort=(combo.reasoning_effort if combo.model in REASONING_MODELS else None),
        chain_of_thought=combo.chain_of_thought,
        note_taking=combo.note_taking
    )

    # Store notes
    current_paper_record["llm_notes"] = notes
    # Store review
    current_paper_record["llm_review"] = review

    return current_paper_record

In [ ]:
# @title Configuration grid

# Complete list of models
MODELS = OPENAI_MODELS + GROQ_MODELS + GEMINI_MODELS

# Generate every single combination
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, CoT, NOTE_TAKING))
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"])

# Conditional deletion whether a model has a temperature or reasoning parameter
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["temperature"]
)

# Drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

# Check and display grid
print(f"Total valid parameter combos: {len(experiment_config)}")

display(experiment_config)

In [ ]:
# @title Subset configuration grid (if active)

if CONFIG_SUBSETTING_ACTIVE:
    original_num_configs = len(experiment_config)

    # Filter by specific models if CONFIG_SUBSET_MODELS is not empty
    if CONFIG_SUBSET_MODELS:
        experiment_config = experiment_config[experiment_config['model'].isin(CONFIG_SUBSET_MODELS)]
        print(f"Configuration grid filtered to models: {CONFIG_SUBSET_MODELS}. Number of configs: {len(experiment_config)}")

    # Filter by specific note_taking options
    if CONFIG_SUBSET_NOTE_TAKING_OPTIONS:
        experiment_config = experiment_config[experiment_config['note_taking'].isin(CONFIG_SUBSET_NOTE_TAKING_OPTIONS)]
        print(f"Configuration grid further filtered by note_taking options: {CONFIG_SUBSET_NOTE_TAKING_OPTIONS}. Number of configs: {len(experiment_config)}")

    # Filter by specific reasoning efforts
    if CONFIG_SUBSET_REASONING_EFFORTS:
        # Handle NaN values for reasoning_effort in non-reasoning models
        experiment_config = experiment_config[
            experiment_config['reasoning_effort'].isna() |
            experiment_config['reasoning_effort'].isin(CONFIG_SUBSET_REASONING_EFFORTS)
        ]
        print(f"Configuration grid further filtered by reasoning_effort: {CONFIG_SUBSET_REASONING_EFFORTS}. Number of configs: {len(experiment_config)}")

    # Filter by specific chain_of_thought options
    if CONFIG_SUBSET_CHAIN_OF_THOUGHTS:
        experiment_config = experiment_config[experiment_config['chain_of_thought'].isin(CONFIG_SUBSET_CHAIN_OF_THOUGHTS)]
        print(f"Configuration grid further filtered by chain_of_thought: {CONFIG_SUBSET_CHAIN_OF_THOUGHTS}. Number of configs: {len(experiment_config)}")

    # Filter by specific temperatures
    if CONFIG_SUBSET_TEMPS:
        # Handle NaN values for temperature in reasoning models
        experiment_config = experiment_config[
            experiment_config['temperature'].isna() |
            experiment_config['temperature'].isin(CONFIG_SUBSET_TEMPS)
        ]
        print(f"Configuration grid further filtered by temperatures: {CONFIG_SUBSET_TEMPS}. Number of configs: {len(experiment_config)}")

    # Group by model and take the first CONFIG_NUM_SUBSET_PER_MODEL rows from each group
    if CONFIG_NUM_SUBSET_PER_MODEL > 0:
        experiment_config = experiment_config.groupby('model').head(CONFIG_NUM_SUBSET_PER_MODEL).reset_index(drop=True)
        print(f"Configuration grid further subsetted to {CONFIG_NUM_SUBSET_PER_MODEL} rows per model.")

    # Check and display subsetted grid
    print(f"Original total valid parameter combos: {original_num_configs}")
    print(f"Final total valid parameter combos after subsetting: {len(experiment_config)}")

    display(experiment_config)
else:
    print("Configuration grid subsetting is inactive.")

In [ ]:
# @title Helpers

### Helpers for logging ###

def _nan_to_none(x):
    """Use pandas' isna to catch float('nan'), numpy.nan, pd.NA, and None"""
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x

def combo_key_tuple(row) -> tuple:
    """Combines run and configuration settings with None instead of NaN"""
    run_signature = row["run_signature"] if "run_signature" in row and pd.notna(row["run_signature"]) else "__legacy__"
    return (
        run_signature,
        row["model"],
        _nan_to_none(row["temperature"]),
        _nan_to_none(row["reasoning_effort"]),
        row["chain_of_thought"],
        row["note_taking"]
    )

def combo_key_str(row) -> str:
    """Creates readable combo string for logging"""
    t = combo_key_tuple(row)
    return f"run={t[0]}|model={t[1]}|temp={t[2]}|re={t[3]}|cot={t[4]}|notes={t[5]}"

# Unique hash for each run
RUN_SIGNATURE = hashlib.sha1(
json.dumps(
    {
        "note_taking_prompt": NOTE_TAKING_PROMPT,
        "review_generation_prompt": REVIEW_GENERATION,
        "max_tokens": MAX_TOKENS
    },
    sort_keys=True,
    ensure_ascii=True,
).encode("utf-8")
).hexdigest()[:12]

### Helpers for output messages ###

def _fmt_combo(combo: ParamCombo) -> str:
    """Helper to format the ParamCombo into a human-readable string"""
    parts = []
    parts.append(f"model={combo.model}")
    if combo.temperature is not None:
        parts.append(f"temperature={combo.temperature}")
    if combo.reasoning_effort is not None:
        parts.append(f"reasoning_effort={combo.reasoning_effort}")
    parts.append(f"chain_of_thought={combo.chain_of_thought}")
    parts.append(f"note_taking={combo.note_taking}")
    return ", ".join(parts)

def log_call(doc_id: str, combo: ParamCombo, **context: Dict[str, Any]):
    """Helper to enrich the ParamCombo string with information for output messages"""
    ctx = ", ".join(f"{k}={v}" for k, v in context.items() if v is not None)
    msg = f"[LLM CALL] paper={doc_id} | {_fmt_combo(combo)}"
    if ctx:
        msg += f" | {ctx}"
    print(msg, flush=True)

## Run

In [ ]:
# @title Review generation

if __name__ == '__main__':
    async def main():

        print(f"Processing and saving intermediate results to: {RESULTS_JSON_PATH}", flush = True)
        print(f"Logging successful iterations to: {LOG_JSON_PATH}", flush = True)

        # Initialize client
        client = AsyncLLMClient(simulate=SIMULATION_ACTIVE, seed=SEED_VALUE)

    ### --- 1. Check for already existing logs and results --- ###

        # Initialize keys
        done_keys = set()

        # Helper to extract relevant columns for combo key creation
        combo_cols = ["run_signature", "model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"]

        # Check and read successful paper-parameter combos from JSON log file
        if os.path.exists(LOG_JSON_PATH):
            try:
                with open(LOG_JSON_PATH, 'r') as f:
                    for line in f:
                        entry = json.loads(line)
                        if 'paper_id' in entry and 'completed_param_combo' in entry:
                            done_keys.add((entry['paper_id'], entry['completed_param_combo']))
                print(f"Loaded {len(done_keys)} completed paper-parameter combo entries from JSON log.")
            except Exception as e:
                print(f"Error reading existing JSON log file: {e}. Starting with empty log.")
                done_keys = set()

        # Check and read successful paper-parameter combos from JSON result file
        if os.path.exists(RESULTS_JSON_PATH):
            try:
                with open(RESULTS_JSON_PATH, 'r') as f:
                    for line in f:
                        entry = json.loads(line)
                        entry_for_combo_key = entry.copy()
                        paper_id = entry["paper_id"]
                        param_combo_str_current = combo_key_str(entry_for_combo_key)
                        done_keys.add((paper_id, param_combo_str_current))

                print(f"Loaded {len(done_keys)} completed paper-parameter combo entries from JSONL results.")

            except Exception as e:
                print(f"Error reading existing JSONL results file: {e}. Moving on.")

        print(f"Total already-completed paper-configuration combos: {len(done_keys)}")

    ### --- 2. Create notes and reviews for all paper-parameter combos --- ###

        all_tasks_to_run = []
        task_metadata = []
        total_combos = len(experiment_config) * len(targets)
        processed_combos = 0

        # Prepare all unique (combo, paper) pairs that need to be processed
        for idx_combo, row_combo in experiment_config.iterrows():
            for idx_paper, paper_data in enumerate(targets):
                paper_id = paper_data["paper_id"]

                # Set string representation of the current combo
                row_combo_for_key = row_combo.copy()
                row_combo_for_key["run_signature"] = RUN_SIGNATURE
                param_combo_str_current = combo_key_str(row_combo_for_key)
                paper_combo_key = (paper_id, param_combo_str_current)

                # Check if this specific paper-combo is already completed
                if paper_combo_key in done_keys:
                    print(f"[SKIP] Paper {paper_id} with combo {param_combo_str_current} already complete.")
                    processed_combos += 1
                    continue

                # Extract current parameters from the configuration grid
                combo = ParamCombo(
                    model=row_combo["model"],
                    temperature=None if pd.isna(row_combo["temperature"]) else float(row_combo["temperature"]),
                    reasoning_effort=None if pd.isna(row_combo["reasoning_effort"]) else str(row_combo["reasoning_effort"]),
                    chain_of_thought=row_combo["chain_of_thought"],
                    note_taking=row_combo["note_taking"]
                )

                # Prepare the setup across the batch
                all_tasks_to_run.append(run_parameter_combo(combo, paper_data, client=client))
                task_metadata.append({
                    "paper_id": paper_id,
                    "param_combo_str": param_combo_str_current,
                    "combo": combo,
                    "paper_data": paper_data
                })

        print(f"Total unique paper-parameter combos to process: {len(all_tasks_to_run)}")

        # Process tasks in batches
        for i in tqdm(range(0, len(all_tasks_to_run), NUM_PARALLEL_BATCH_SIZE), desc="Processing Batches"):
            batch_tasks = all_tasks_to_run[i:i + NUM_PARALLEL_BATCH_SIZE]
            batch_metadata = task_metadata[i:i + NUM_PARALLEL_BATCH_SIZE]

            print(f"\n[RUN BATCH] Processing {len(batch_tasks)} tasks concurrently.", flush=True)

            results = await asyncio.gather(*batch_tasks, return_exceptions=True)

            # Initialize result and log dic for current batch
            batch_results_dicts = []
            batch_log_dicts = []

            # Extract information from each result in the batch
            for j, result in enumerate(results):
                metadata = batch_metadata[j]
                paper_id = metadata["paper_id"]
                param_combo_str = metadata["param_combo_str"]
                combo_obj = metadata["combo"]

                # Initialize result record
                record_for_results = None

                if isinstance(result, Exception):
                    print(f"[ERROR BATCH] Paper {paper_id} with combo {param_combo_str} -> {type(result).__name__}: {result}", flush=True)

                    # Create a dictionary record for results with error status
                    error_record = {
                        "paper_id": paper_id,
                        "model": combo_obj.model,
                        "temperature": combo_obj.temperature,
                        "reasoning_effort": combo_obj.reasoning_effort,
                        "chain_of_thought": combo_obj.chain_of_thought,
                        "note_taking": combo_obj.note_taking,
                        "run_signature": RUN_SIGNATURE
                    }
                    # Populate llm_notes and llm_review columns with error messages
                    error_record["llm_notes"] = NOTE_TAKING_FAILURE
                    error_record["llm_review"] = REVIEW_GENERATION_FAILURE
                    record_for_results = error_record

                else:
                    # Append result to the record
                    record_for_results = result

                # Always append the constructed result record to the batch list
                batch_results_dicts.append(record_for_results)

                # Always add to done_keys to mark this combo as processed/attempted
                done_keys.add((paper_id, param_combo_str))

                # Prepare and append log entry for every processed combo
                new_log_entry = {
                    'paper_id': paper_id,
                    'completed_param_combo': param_combo_str
                }
                batch_log_dicts.append(new_log_entry)

            # After processing all results in the batch, append them at once to JSONL files
            if batch_results_dicts:
                with open(RESULTS_JSON_PATH, 'a') as f:
                    for record in batch_results_dicts:
                        f.write(json.dumps(record) + '\n')
                print(f"Appended {len(batch_results_dicts)} results to {RESULTS_JSON_PATH}")

            if batch_log_dicts:
                with open(LOG_JSON_PATH, 'a') as f:
                    for record in batch_log_dicts:
                        f.write(json.dumps(record) + '\n')
                print(f"Appended {len(batch_log_dicts)} log entries to {LOG_JSON_PATH}")


    await main() # Run the async main function

## Transformation

In [ ]:
# @title Conversion

# Convert RESULTS_JSON_PATH to RESULTS_PATH (Parquet)
if os.path.exists(RESULTS_JSON_PATH):
    try:
        print(f"Converting {RESULTS_JSON_PATH} to {RESULTS_PATH}...")
        jsonl_df = pd.read_json(RESULTS_JSON_PATH, lines=True)
        if not jsonl_df.empty:
            jsonl_df.to_parquet(RESULTS_PATH, index=False)
            print(f"Successfully converted and saved results to {RESULTS_PATH}.")
        else:
            print(f"{RESULTS_JSON_PATH} is empty. No Parquet file generated for results.")
    except Exception as e:
        print(f"Error converting results JSONL to Parquet: {e}")
else:
    print(f"Intermediate results file '{RESULTS_JSON_PATH}' does not exist. No results Parquet file generated.")

# Convert LOG_JSON_PATH to LOG_PATH (Parquet)
if os.path.exists(LOG_JSON_PATH):
    try:
        print(f"Converting {LOG_JSON_PATH} to {LOG_PATH}...")
        jsonl_log_df = pd.read_json(LOG_JSON_PATH, lines=True)
        if not jsonl_log_df.empty:
            jsonl_log_df.to_parquet(LOG_PATH, index=False)
            print(f"Successfully converted and saved log to {LOG_PATH}.")
        else:
            print(f"'{LOG_JSON_PATH}' is empty. No Parquet file generated for log.")
    except Exception as e:
        print(f"Error converting log JSONL to Parquet: {e}")
else:
    print(f"Intermediate log file '{LOG_JSON_PATH}' does not exist. No log Parquet file generated.")

In [ ]:
# @title Results

# Load result file
if os.path.exists(RESULTS_PATH):
    try:
        llm_results_df = pd.read_parquet(RESULTS_PATH)
        # Check and display results
        display(llm_results_df.shape)
        display(llm_results_df)

    except:
        print(f"Warning: Could not read existing results parquet file {RESULTS_PATH}.")
else:
    print(f"Results file '{RESULTS_PATH}' does not exist.")

# Load log file
if os.path.exists(LOG_PATH):
    try:
        llm_log_df = pd.read_parquet(LOG_PATH)
        # Check and display log
        display(llm_log_df.shape)
        display(llm_log_df.head(5))

    except:
        print(f"Warning: Could not read existing log parquet file {LOG_PATH}.")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")